In [ ]:

import pandas as pd
import numpy as np
import holidays
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split


In [16]:

# Step 2: load only needed columns, dayfirst parsing
df = pd.read_csv(
    r"Prod_data\tbl_device_history.csv",
    usecols=["trolleyId", "geozoneId", "createdOn", "status"],
    parse_dates=["createdOn"],
    dayfirst=True
)

# keep only status==1 (active) if relevant, drop geozone 0 (unknown)
df = df[(df["status"] == 1) & (df["geozoneId"] != 0)].copy()
df = df[["trolleyId", "geozoneId", "createdOn"]]


In [7]:
df['geozoneId'].unique()

array([11,  2,  4, 16,  3,  5,  7,  6,  8,  9, 10, 13, 14, 15, 17,  1, 18,
       12, 20, 21, 22, 24])

In [17]:

# Step 3: dedup - one record per trolley/zone/hour
df = df.sort_values(["trolleyId", "geozoneId", "createdOn"])
df["date"] = df["createdOn"].dt.date
df["hour"] = df["createdOn"].dt.hour



In [18]:
df.info()

<class 'pandas.DataFrame'>
Index: 289313 entries, 588 to 299024
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   trolleyId  289313 non-null  int64         
 1   geozoneId  289313 non-null  int64         
 2   createdOn  289313 non-null  datetime64[us]
 3   date       289313 non-null  object        
 4   hour       289313 non-null  int32         
dtypes: datetime64[us](1), int32(1), int64(2), object(1)
memory usage: 12.1+ MB


In [19]:
df

,trolleyId,geozoneId,createdOn,date,hour
588,1,1,2026-03-04 16:25:00,2026-03-04,16
597,1,1,2026-03-04 16:28:00,2026-03-04,16
608,1,1,2026-03-04 16:30:00,2026-03-04,16
616,1,1,2026-03-04 16:32:00,2026-03-04,16
626,1,1,2026-03-04 16:36:00,2026-03-04,16
...,...,...,...,...,...
281834,501,18,2026-05-09 06:09:00,2026-05-09,6
282000,501,18,2026-05-09 06:34:00,2026-05-09,6
167609,501,20,2026-04-30 14:15:00,2026-04-30,14
298941,501,24,2026-05-11 04:17:00,2026-05-11,4


In [20]:
df = df.drop_duplicates(subset=["trolleyId", "geozoneId", "date", "hour"])


In [21]:
df.info()

<class 'pandas.DataFrame'>
Index: 102084 entries, 588 to 298941
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   trolleyId  102084 non-null  int64         
 1   geozoneId  102084 non-null  int64         
 2   createdOn  102084 non-null  datetime64[us]
 3   date       102084 non-null  object        
 4   hour       102084 non-null  int32         
dtypes: datetime64[us](1), int32(1), int64(2), object(1)
memory usage: 4.3+ MB


In [22]:
df

,trolleyId,geozoneId,createdOn,date,hour
588,1,1,2026-03-04 16:25:00,2026-03-04,16
3582,1,1,2026-03-07 10:48:00,2026-03-07,10
3592,1,1,2026-03-07 11:02:00,2026-03-07,11
3635,1,1,2026-03-07 12:03:00,2026-03-07,12
4126,1,1,2026-03-07 22:39:00,2026-03-07,22
...,...,...,...,...,...
280891,501,18,2026-05-09 04:07:00,2026-05-09,4
281378,501,18,2026-05-09 05:07:00,2026-05-09,5
281834,501,18,2026-05-09 06:09:00,2026-05-09,6
167609,501,20,2026-04-30 14:15:00,2026-04-30,14


In [23]:
# Step 4: aggregate -> unique trolley count per zone/date/hour
agg = df.groupby(["date", "hour", "geozoneId"])["trolleyId"].nunique().reset_index()
agg.columns = ["date", "hour", "geozoneId", "trolley_count"]


In [28]:
agg

,date,hour,geozoneId,trolley_count
0,2026-03-04,13,2,3
1,2026-03-04,13,4,1
2,2026-03-04,13,11,10
3,2026-03-04,13,16,1
4,2026-03-04,14,2,10
...,...,...,...,...
6704,2026-05-11,4,16,38
6705,2026-05-11,4,17,27
6706,2026-05-11,4,18,53
6707,2026-05-11,4,20,1


In [ ]:


# # Build complete grid (date x hour x geozoneId) so missing slots become 0
# all_dates = pd.date_range(df["date"].min(), df["date"].max(), freq="D").date
# all_hours = range(24)
# all_zones = sorted(df["geozoneId"].unique())

# grid = pd.MultiIndex.from_product(
#     [all_dates, all_hours, all_zones],
#     names=["date", "hour", "geozoneId"]
# ).to_frame(index=False)

# full = grid.merge(agg, on=["date", "hour", "geozoneId"], how="left")
# full["trolley_count"] = full["trolley_count"].fillna(0).astype(int)

# full["datetime"] = pd.to_datetime(full["date"]) + pd.to_timedelta(full["hour"], unit="h")
# full = full.sort_values(["geozoneId", "datetime"]).reset_index(drop=True)


In [26]:
agg

,date,hour,geozoneId,trolley_count
0,2026-03-04,13,2,3
1,2026-03-04,13,4,1
2,2026-03-04,13,11,10
3,2026-03-04,13,16,1
4,2026-03-04,14,2,10
...,...,...,...,...
6704,2026-05-11,4,16,38
6705,2026-05-11,4,17,27
6706,2026-05-11,4,18,53
6707,2026-05-11,4,20,1


In [27]:
agg['trolley_count'].unique

<bound method Series.unique of 0        3
1        1
2       10
3        1
4       10
        ..
6704    38
6705    27
6706    53
6707     1
6708     5
Name: trolley_count, Length: 6709, dtype: int64>

In [33]:

# Step 5: time features
agg['date'] = pd.to_datetime(agg['date'], errors='coerce')
agg["month"] = agg["date"].dt.month
agg["day"] = agg["date"].dt.day
agg["day_of_week"] = agg["date"].dt.dayofweek
agg["is_weekend"] = agg["date"].isin([5, 6]).astype(int)
agg["is_month_start"] = agg["date"].dt.is_month_start.astype(int)
agg["is_month_end"] = agg["date"].dt.is_month_end.astype(int)

# peak hour flags (adjust to your domain, e.g. 7-9 and 17-19)
agg["is_morning_peak"] = agg["hour"].between(7, 9).astype(int)
agg["is_evening_peak"] = agg["hour"].between(17, 19).astype(int)

# cyclical encodings
agg["hour_sin"] = np.sin(2 * np.pi * agg["hour"] / 24)
agg["hour_cos"] = np.cos(2 * np.pi * agg["hour"] / 24)
agg["month_sin"] = np.sin(2 * np.pi * agg["month"] / 12)
agg["month_cos"] = np.cos(2 * np.pi * agg["month"] / 12)
agg["dow_sin"] = np.sin(2 * np.pi * agg["day_of_week"] / 7)
agg["dow_cos"] = np.cos(2 * np.pi * agg["day_of_week"] / 7)


In [34]:
agg

,date,hour,geozoneId,trolley_count,month,day,day_of_week,is_weekend,is_month_start,is_month_end,is_morning_peak,is_evening_peak,hour_sin,hour_cos,month_sin,month_cos,dow_sin,dow_cos
0,2026-03-04,13,2,3,3,4,2,0,0,0,0,0,-0.258819,-0.965926,1.0,6.123234e-17,0.974928,-0.222521
1,2026-03-04,13,4,1,3,4,2,0,0,0,0,0,-0.258819,-0.965926,1.0,6.123234e-17,0.974928,-0.222521
2,2026-03-04,13,11,10,3,4,2,0,0,0,0,0,-0.258819,-0.965926,1.0,6.123234e-17,0.974928,-0.222521
3,2026-03-04,13,16,1,3,4,2,0,0,0,0,0,-0.258819,-0.965926,1.0,6.123234e-17,0.974928,-0.222521
4,2026-03-04,14,2,10,3,4,2,0,0,0,0,0,-0.500000,-0.866025,1.0,6.123234e-17,0.974928,-0.222521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6704,2026-05-11,4,16,38,5,11,0,0,0,0,0,0,0.866025,0.500000,0.5,-8.660254e-01,0.000000,1.000000
6705,2026-05-11,4,17,27,5,11,0,0,0,0,0,0,0.866025,0.500000,0.5,-8.660254e-01,0.000000,1.000000
6706,2026-05-11,4,18,53,5,11,0,0,0,0,0,0,0.866025,0.500000,0.5,-8.660254e-01,0.000000,1.000000
6707,2026-05-11,4,20,1,5,11,0,0,0,0,0,0,0.866025,0.500000,0.5,-8.660254e-01,0.000000,1.000000


# Malaysia public holidays


In [ ]:

my_holidays = holidays.Malaysia(years=range(agg["date"].dt.year.min(), agg["date"].dt.year.max() + 1))

In [37]:
my_holidays

{datetime.date(2026, 2, 17): 'Tahun Baharu Cina', datetime.date(2026, 2, 18): 'Tahun Baharu Cina (Hari Kedua)', datetime.date(2026, 5, 31): 'Hari Wesak', datetime.date(2026, 5, 1): 'Hari Pekerja', datetime.date(2026, 6, 1): 'Hari Keputeraan Rasmi Seri Paduka Baginda Yang di-Pertuan Agong', datetime.date(2026, 8, 31): 'Hari Kebangsaan', datetime.date(2026, 9, 16): 'Hari Malaysia', datetime.date(2026, 12, 25): 'Hari Krismas', datetime.date(2026, 6, 17): 'Awal Muharam', datetime.date(2026, 8, 25): 'Hari Keputeraan Nabi Muhammad S.A.W.', datetime.date(2026, 3, 21): 'Hari Raya Puasa', datetime.date(2026, 3, 22): 'Hari Raya Puasa (Hari Kedua)', datetime.date(2026, 5, 27): 'Hari Raya Qurban', datetime.date(2026, 3, 23): 'Cuti Hari Raya Puasa (Hari Kedua)', datetime.date(2026, 6, 2): 'Cuti Hari Wesak'}

In [38]:

agg["is_holiday"] = agg["date"].apply(lambda d: 1 if d in my_holidays else 0)

# day before / after holiday (often affects traffic too)
holiday_dates = set(my_holidays.keys())
agg["is_day_before_holiday"] = agg["date"].apply(lambda d: 1 if (d + pd.Timedelta(days=3)) in holiday_dates else 0)
agg["is_day_after_holiday"] = agg["date"].apply(lambda d: 1 if (d - pd.Timedelta(days=3)) in holiday_dates else 0)


In [42]:
agg['is_holiday'].value_counts()

is_holiday
0    6432
1     277
Name: count, dtype: int64

In [ ]:

# # Lag features - per geozone, grid is complete so shift is safe
# full = agg.sort_values(["geozoneId", "date"]).reset_index(drop=True)

# group = full.groupby("geozoneId")["trolley_count"]

# full["lag_1h"]   = group.shift(1)
# full["lag_2h"]   = group.shift(2)
# full["lag_3h"]   = group.shift(3)
# full["lag_24h"]  = group.shift(24)        # same hour yesterday
# full["lag_168h"] = group.shift(24 * 7)    # same hour last week

# # rolling stats (based on past values only, shift(1) first to avoid leakage)
# shifted = group.shift(1)
# full["rolling_mean_3h"]  = shifted.rolling(3).mean().reset_index(level=0, drop=True)
# full["rolling_mean_24h"] = shifted.rolling(24).mean().reset_index(level=0, drop=True)
# full["rolling_std_24h"]  = shifted.rolling(24).std().reset_index(level=0, drop=True)

# # drop rows where lag/rolling windows aren't filled yet (start of series)
# full = full.dropna().reset_index(drop=True)


In [44]:
full

,date,hour,geozoneId,trolley_count,month,day,day_of_week,is_weekend,is_month_start,is_month_end,...,is_day_before_holiday,is_day_after_holiday,lag_1h,lag_2h,lag_3h,lag_24h,lag_168h,rolling_mean_3h,rolling_mean_24h,rolling_std_24h
0,2026-03-16,0,1,3,3,16,0,0,0,0,...,0,0,3.0,2.0,1.0,3.0,4.0,2.000000,1.958333,1.458980
1,2026-03-16,1,1,3,3,16,0,0,0,0,...,0,0,3.0,3.0,2.0,1.0,10.0,2.666667,1.958333,1.458980
2,2026-03-16,2,1,4,3,16,0,0,0,0,...,0,0,3.0,3.0,3.0,2.0,1.0,3.000000,2.041667,1.458980
3,2026-03-16,3,1,1,3,16,0,0,0,0,...,0,0,4.0,3.0,3.0,1.0,2.0,3.333333,2.125000,1.512628
4,2026-03-16,4,1,1,3,16,0,0,0,0,...,0,0,1.0,4.0,3.0,1.0,3.0,2.666667,2.125000,1.512628
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3945,2026-05-10,6,24,2,5,10,6,0,0,0,...,0,0,2.0,1.0,1.0,1.0,6.0,1.333333,1.583333,1.017955
3946,2026-05-10,7,24,2,5,10,6,0,0,0,...,0,0,2.0,2.0,1.0,1.0,13.0,1.666667,1.625000,1.013496
3947,2026-05-10,10,24,1,5,10,6,0,0,0,...,0,0,2.0,2.0,2.0,1.0,8.0,2.000000,1.666667,1.007220
3948,2026-05-11,3,24,5,5,11,0,0,0,0,...,0,0,1.0,2.0,2.0,2.0,3.0,1.666667,1.666667,1.007220


In [81]:

# Step 6: time-based split (no shuffle)
full = full.sort_values("date").reset_index(drop=True)

full["day"] = full["date"].dt.day      # date-of-month (1-31)
full["month_num"] = full["date"].dt.month
full["year"] = full["date"].dt.year

cutoff = full["date"].max() - pd.Timedelta(days=14)
train = full[full["date"] <= cutoff]
test  = full[full["date"] > cutoff]

# feature_cols = [[
#     "geozoneId", "hour", "day", "month_num", "year", "day_of_week", "is_weekend",
#     "is_month_start", "is_month_end",
#     "is_morning_peak", "is_evening_peak",
#     "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
#     "is_holiday", "is_day_before_holiday", "is_day_after_holiday"
# ]]
# target_col = [["trolley_count"]]




In [86]:
X = full[["geozoneId", "hour", "day", "month_num", "year", "day_of_week", "is_weekend",
    "is_month_start", "is_month_end",
    "is_morning_peak", "is_evening_peak",
    "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
    "is_holiday", "is_day_before_holiday", "is_day_after_holiday"]]
y = full[['trolley_count']]


In [87]:
X.head()

,geozoneId,hour,day,month_num,year,day_of_week,is_weekend,is_month_start,is_month_end,is_morning_peak,is_evening_peak,hour_sin,hour_cos,month_sin,month_cos,dow_sin,dow_cos,is_holiday,is_day_before_holiday,is_day_after_holiday
0,18,17,14,3,2026,5,0,0,0,0,1,-0.965926,-2.588190e-01,1.0,6.123234e-17,-0.974928,-0.222521,0,0,0
1,16,2,14,3,2026,5,0,0,0,0,0,0.500000,8.660254e-01,1.0,6.123234e-17,-0.974928,-0.222521,0,0,0
2,18,14,14,3,2026,5,0,0,0,0,0,-0.500000,-8.660254e-01,1.0,6.123234e-17,-0.974928,-0.222521,0,0,0
3,18,5,14,3,2026,5,0,0,0,0,0,0.965926,2.588190e-01,1.0,6.123234e-17,-0.974928,-0.222521,0,0,0
4,18,6,14,3,2026,5,0,0,0,0,0,1.000000,6.123234e-17,1.0,6.123234e-17,-0.974928,-0.222521,0,0,0


In [88]:
y

,trolley_count
0,1
1,13
2,2
3,7
4,6
...,...
3945,2
3946,12
3947,2
3948,71


In [89]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [90]:

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

model = DecisionTreeRegressor(
    max_depth=6,
    min_samples_leaf=20,
    random_state=42
)
model.fit(X_train, y_train)

preds = model.predict(X_test)
preds = np.clip(preds, 0, None)  # counts can't be negative

mae = mean_absolute_error(y_test, preds)
#rmse = mean_squared_error(y_test, preds, squared=False)
r2 = r2_score(y_test, preds)

print(f"MAE: {mae:.3f}")
#print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")


MAE: 7.382
R2: 0.829


In [92]:
feature_cols = [
     "geozoneId", "hour", "day", "month_num", "year", "day_of_week", "is_weekend",
     "is_month_start", "is_month_end",
     "is_morning_peak", "is_evening_peak",
     "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
     "is_holiday", "is_day_before_holiday", "is_day_after_holiday"
]

In [93]:

importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)


geozoneId                0.596779
month_cos                0.244222
day                      0.148867
dow_sin                  0.005608
month_sin                0.002155
hour_cos                 0.001276
dow_cos                  0.000810
hour_sin                 0.000107
day_of_week              0.000102
month_num                0.000065
hour                     0.000009
is_weekend               0.000000
is_evening_peak          0.000000
is_morning_peak          0.000000
is_month_end             0.000000
is_month_start           0.000000
year                     0.000000
is_holiday               0.000000
is_day_before_holiday    0.000000
is_day_after_holiday     0.000000
dtype: float64


In [100]:
X_train

,geozoneId,hour,day,month_num,year,day_of_week,is_weekend,is_month_start,is_month_end,is_morning_peak,is_evening_peak,hour_sin,hour_cos,month_sin,month_cos,dow_sin,dow_cos,is_holiday,is_day_before_holiday,is_day_after_holiday
3769,9,22,10,5,2026,6,0,0,0,0,0,-0.500000,8.660254e-01,0.500000,-0.866025,-0.781831,0.623490,0,0,0
1407,18,4,26,4,2026,6,0,0,0,0,0,0.866025,5.000000e-01,0.866025,-0.500000,-0.781831,0.623490,0,0,0
2775,1,16,5,5,2026,1,0,0,0,0,0,-0.866025,-5.000000e-01,0.500000,-0.866025,0.781831,0.623490,0,0,0
3341,18,22,8,5,2026,4,0,0,0,0,0,-0.500000,8.660254e-01,0.500000,-0.866025,-0.433884,-0.900969,0,0,0
1116,11,9,23,4,2026,3,0,0,0,1,0,0.707107,-7.071068e-01,0.866025,-0.500000,0.433884,-0.900969,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1130,11,0,23,4,2026,3,0,0,0,0,0,0.000000,1.000000e+00,0.866025,-0.500000,0.433884,-0.900969,0,0,0
1294,11,14,25,4,2026,5,0,0,0,0,0,-0.500000,-8.660254e-01,0.866025,-0.500000,-0.974928,-0.222521,0,0,0
860,11,4,13,4,2026,0,0,0,0,0,0,0.866025,5.000000e-01,0.866025,-0.500000,0.000000,1.000000,0,0,0
3507,1,21,9,5,2026,5,0,0,0,0,0,-0.707107,7.071068e-01,0.500000,-0.866025,-0.974928,-0.222521,0,0,0


In [101]:
print("Total rows:", len(full))
print("Date range:", full["date"].min(), "to", full["date"].max())
print("Unique dates:", full["date"].nunique())
print()
print("Train rows:", len(X_train), "| dates:", X_train["day"].nunique())
print("Test rows:", len(X_test), "| dates:", X_test["day"].nunique())
print()
print("Train trolley_count stats:")
print(y_train["trolley_count"].describe())
print()
print("Test trolley_count stats:")
print(y_test["trolley_count"].describe())
print()
print("Zones in train:", sorted(X_train["geozoneId"].unique()))
print("Zones in test:", sorted(X_test["geozoneId"].unique()))

Total rows: 3950
Date range: 2026-03-14 00:00:00 to 2026-05-11 00:00:00
Unique dates: 52

Train rows: 3160 | dates: 31
Test rows: 790 | dates: 31

Train trolley_count stats:
count    3160.000000
mean       21.033544
std        37.476773
min         1.000000
25%         2.000000
50%         3.000000
75%        16.000000
max       209.000000
Name: trolley_count, dtype: float64

Test trolley_count stats:
count    790.000000
mean      19.036709
std       36.539541
min        1.000000
25%        2.000000
50%        3.500000
75%       14.000000
max      202.000000
Name: trolley_count, dtype: float64

Zones in train: [np.int64(1), np.int64(3), np.int64(5), np.int64(7), np.int64(8), np.int64(9), np.int64(11), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(20), np.int64(24)]
Zones in test: [np.int64(1), np.int64(3), np.int64(5), np.int64(7), np.int64(8), np.int64(9), np.int64(11), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(20),